In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime
import  time
from rich import print
class OverAllState(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str
# runtime.stream_writer でストリーミング出力内容を実現する
def node_a(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("ノード A を実行中...")
    time.sleep(2)
    return {
        "node_a_output": "ノードAの出力"
    }

def node_b(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("ノード B を実行中...")
    time.sleep(2)
    return {
        "node_b_output": "ノードBの出力"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
for chunk in graph.stream(
    {"initial_state": "初期状態"},
    #ノードのカスタムストリーミング出力を有効にするには stream_mode=["custom"] を追加する必要がある
    stream_mode=["custom"],
):
    print(chunk)

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt.tool_node import ToolNode, ToolRuntime
from langgraph.runtime import Runtime
from langchain.tools import tool

from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

@tool(parse_docstring=True)
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """
    都市に基づいて当日の天気を照会する

    Args:
        city: 都市名
    """
    stream_writer = runtime.stream_writer
    stream_writer(f"{city} の今日の天気を照会中...")
    return f"{city} は今日良い天気です"

tools = [get_weather]
model_with_tools = model.bind_tools(tools=tools)

def llm_node(state: MessagesState, runtime: Runtime) -> MessagesState:
    messages = state["messages"]
    response = model_with_tools.invoke(messages)

    stream_writer = runtime.stream_writer
    stream_writer("llm_node を実行中...")

    return {
        "messages": [response]
    }

def router(state: MessagesState) -> Literal["tool_node", END]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        return "tool_node"
    return END

builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", ToolNode(tools=tools))
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, path_map=["tool_node", END])
builder.add_edge("tool_node", "llm_node")

graph = builder.compile()

for chunk in graph.stream(
    {"messages": [HumanMessage("今日の東京の天気はどうですか？")]},
    # カスタムストリーミング出力を行うには custom を追加する必要がある
    stream_mode=["custom"]
):
    print(chunk)